In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("../data/raw/superstore.csv", encoding="latin1")#UTF encoding not followed here as file likely from excel hence used latin1 encoding which is more flexible

In [ ]:
df.head()
df.columns


In [ ]:
df.info()
df.describe()

In [62]:
df['Order Date']=pd.to_datetime(df['Order Date'])#Converting the date text to date object which can be processed by the ML model.
df=df.set_index("Order Date")
daily_sales=df.groupby('Order Date')['Sales'].sum().reset_index()#New dataset which is time-series ready, mapping date to daily sales.
#Also reset-index converts it to a proper dataframe format easier for plotting and ML understanding 
daily_sales=daily_sales.sort_values('Order Date')

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(daily_sales['Order Date'],daily_sales['Sales'])
plt.title("Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

In [ ]:
#Now data smoothing
daily_sales['Sales_Smooth'] = daily_sales['Sales'].rolling(window=7).mean()#Window=7 means instead of individual columns, it creates a sliding window of 7 days where suppose value of day 1 is replaced by average of days 1-7 similarly for other days as well and .mean() replaces it with average.
#We did above smoothing to remove random spikes
#Now plotting smooted data
plt.figure(figsize=(12,5))
plt.plot(daily_sales['Order Date'], daily_sales['Sales'], alpha=0.3, label="Raw Sales")#alpha=0.3 to decrease intensity of this plot and stress more on other plot.
plt.plot(daily_sales['Order Date'], daily_sales['Sales_Smooth'], color='red', label="7-day Moving Avg")
plt.title("Sales Trend (Smoothed)")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.show()

In [ ]:
#Monthly trend 
monthly_sales = df.copy()
monthly_sales['Order Date'] = pd.to_datetime(monthly_sales['Order Date'])
monthly_sales = monthly_sales.set_index('Order Date')#Needed to be done for time series data
monthly_sales = monthly_sales.resample('M')['Sales'].sum().reset_index()
#Resample 'M' means group by MONTH now rather than day, can't use group by as 'M' is not a separate column.
#Use 'resample' as it 'groups time intervals' and it is used for time series data.

In [ ]:
#Plotting monthly trend
plt.figure(figsize=(12,5))
plt.plot(monthly_sales['Order Date'], monthly_sales['Sales'])
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.show()
#Monthly graph shows strong seasonality towards end of the year months and also high variability in sales but this graph indicates overall sales growth.

In [ ]:
monthly_pattern=daily_sales.groupby("Month")['Sales'].mean()
print(monthly_pattern)
#Helps understand Seasonality

In [71]:
#Apply Feature Engineering
#Feauture Engineering - Converting one column into several useful features, like the order date column is being used to get new columns like year, month, day, weekday, etc.
# 'dt' is a panda property used to get access to properties of dates
df['Order Date']=pd.to_datetime(df['Order Date'])
df['Year']=df['Order Date'].dt.year #Needed to understand TREND
df['Month']=df['Order Date'].dt.month #Needed to understand seasonal behaviour
df['Day']=df['Order Date'].dt.day
df['Weekday']=df['Order Date'].dt.weekday


In [ ]:
#Now we want to work on our Daily Sales dataset so lets apply feature extraction it.
#Also year day month etc. are generally not enough for forecasting so we add another column called 'Lag Sales' which is a record of the previous day's sales.
daily_sales['Year']=daily_sales['Order Date'].dt.year
daily_sales['Month']=daily_sales['Order Date'].dt.month
daily_sales['Day']=daily_sales['Order Date'].dt.day
daily_sales['Weekday']=daily_sales['Order Date'].dt.weekday
daily_sales['Lag_1']=daily_sales['Sales'].shift(1)
daily_sales['Lag_7']=daily_sales['Sales'].shift(7)
daily_sales['Lag_30']=daily_sales['Sales'].shift(30)


In [119]:
#We can't train ML model on NaN values hence we drop the rows with such values as losing only a few rows from such a large dataset is insignificant.
daily_sales.dropna(inplace=True)
#We can also write daily_sales=daily_sales.dropna()

In [ ]:
daily_sales.isna().sum()

In [ ]:
daily_sales.shape #Tells about how many days of data is available
daily_sales.info() #Helps us understand if our data table has been made properly
print("Start date: " + str(daily_sales['Order Date'].min()))#Start date of our data
print("End date: " + str(daily_sales['Order Date'].max()))#End date of our data
#these 2 let us know how many years worth of data we have. We have approx. 3 to 4 years of data in our case which is good for forecasting.

In [ ]:
plt.figure(figsize=(12,5))
plt.hist(daily_sales['Sales'],bins=30)
plt.title("Distribution of daily sales")
plt.show()
#We get a histogram with very long right tail which means occasional huge sale spikes, forecasting becomes difficult.

In [ ]:
daily_sales[['Sales','Lag_1']].corr() #Helps to understand correlation between daily sales and previous day sales, i.e. does yesterday affect today.
#Here correlation value is less about 0.1 which means yesterday sales and today sales have less relation, or yesterday's sales have less effect on today's sales.

In [ ]:
daily_sales.describe() #Tells us about our overall data distribution, helps in understanding sales spikes, etc.

#Overall we can conclude our data is 'Right-Skewed'.

In [ ]:
#As our data has huge spikes, huge variabilities we can't use yesterday's sales alone to predict future sales.
#We will incorporate more features in next file to better the forecasting done by the model.
'''Dataset Summary
1231 daily sales records
Nearly 4 years of historical data
No missing values after preprocessing
Sales Behavior
Average daily sales ≈ $1,862
Median daily sales ≈ $1,071
Significant sales spikes exist
Highest sales day ≈ $28,106
Lowest sales day ≈ $2
Forecasting Challenges
High variability in sales
Weak relationship with previous day's sales
Additional seasonal and trend features will be required'''

In [121]:
daily_sales.to_csv(
    "../data/processed/daily_sales_processed.csv",
    index=False
)
#Puts the processed dataset daily_sales into our processed folder